# Example 3 - Notebook

In [ ]:
import os
from pathlib import Path

# force the base dir to the current working directory
BASE_DIR = Path.cwd().resolve()
os.environ['TSLIES_DIR'] = str(BASE_DIR)

In [ ]:
from tslies.background.bnnpredictor import BNNPredictor
from tslies.trigger import Trigger
from tslies.utils import Data, File
from tslies.plotter import Plotter

try:
    from example_config import y_cols, y_pred_cols, x_cols, x_cols_excluded, units, latex_y_cols, thresholds
    from catalogs import CatalogsReader
except ModuleNotFoundError:
    from tslies.examples.example3.example_config import (
        y_cols,
        y_pred_cols,
        x_cols,
        x_cols_excluded,
        units,
        latex_y_cols,
        thresholds,
    )
    from tslies.examples.example3.catalogs import CatalogsReader


In [ ]:
catalog = CatalogsReader().catalog_df
print(catalog.head())

In [ ]:
x_cols = [col for col in x_cols if col not in x_cols_excluded]
inputs_outputs_df = File().read_dfs_from_weekly_pk_folder(start=0, stop=1000)
print(inputs_outputs_df.head())
# inputs_outputs_df = Data.get_masked_dataframe(data=inputs_outputs_df, start=0, stop=2000, column='index')
Plotter(df = inputs_outputs_df[[col for col in y_cols if 'Xpos' in col] + [f'{col}_smooth' for col in y_cols if 'Xpos' in col] + ['GOES_XRSA_HARD', 'GOES_XRSB_SOFT', 'MET', 'datetime']], label = 'Inputs and outputs').df_plot_tiles(x_col = 'datetime', excluded_cols = [], y_cols=y_cols, latex_y_cols=latex_y_cols, show = True, smoothing_key='smooth')


In [ ]:
"""Runs the neural network model."""
nn = BNNPredictor(inputs_outputs_df, y_cols, x_cols, y_pred_cols, latex_y_cols, units, False)
hyperparams_combinations = {
    "units_for_layers": ([90], [90], [90], [70], [50]),
    "epochs": [6],
    "bs": [1000],
    "do": [0.02],
    "norm": [0],
    "drop": [0],
    "opt_name": ["Adam"],
    "lr": [0.001],
    "metrics": ["negative_log_likelihood_var+mae_bnn+spectral_loss_bnn"],
    "loss_type": ["negative_log_likelihood_var+mae_bnn+spectral_loss_bnn"],
}

for params in nn.get_hyperparams_combinations(hyperparams_combinations, use_previous=False):
    nn.set_hyperparams(params)
    nn.create_model()
    nn.train()
    nn.update_summary()
    Plotter.save(params)


In [ ]:
y_pred = None
if y_pred is None or len(y_pred) == 0:
    batch_size = len(inputs_outputs_df)
    for i in range(0, len(inputs_outputs_df), batch_size):
        _, y_pred = nn.predict(
            start=i,
            end=i + batch_size,
            write_bkg=True,
            num_batches=1,
            save_predictions_plot=False,
        )

tiles_df = Data.merge_dfs(inputs_outputs_df, y_pred)

for face, face_pred in zip(y_cols, y_pred_cols):
    tiles_df[f"{face}_norm"] = (tiles_df[face] - tiles_df[face_pred]) / tiles_df[f"{face}_std"]

reset = tiles_df["MET"].diff() > 60
trigger = Trigger(
    tiles_df,
    y_cols,
    y_pred_cols,
    thresholds=thresholds,
    trigger_type="focus",
    units=units,
    latex_y_cols=latex_y_cols,
)
trigger.run(reset_condition=reset)
merged_anomalies, return_df = trigger.identify_and_merge_triggers(merge_interval=60)
detections_df = trigger.get_detections_df(["MET"])
print(detections_df)
trigger.save_detections_csv(detections_df=detections_df)
detections_df, filtered_anomalies = trigger.filter_from_catalog(catalog, merged_anomalies, detections_df)
print(detections_df)
trigger.save_detections_csv(detections_df=detections_df, suffix="_in_catalog")
trigger.plot_anomalies(
    filtered_anomalies,
    return_df,
    support_vars=["GOES_XRSA_HARD_EARTH_OCCULTED"],
)


In [ ]:
import pandas as pd

tiles_df = inputs_outputs_df.copy()
stats = []
for face in y_cols:
    tiles_df[f"{face}_std"] = tiles_df[face].rolling(window=120, center=True, min_periods=1).std()
    tiles_df[f"{face}_pred"] = tiles_df[face].rolling(window=120, center=True, min_periods=1).mean()

    norm_face = (tiles_df[face] - tiles_df[f"{face}_pred"]) / tiles_df[f"{face}_std"]
    stats.append(
        {
            "face": face,
            "std": round(norm_face.std(), 3),
            "mean": round(norm_face.mean(), 3),
        }
    )

stats_df = pd.DataFrame(stats)
print(stats_df.set_index("face").T.to_string(header=True))

reset = tiles_df["MET"].diff() > 60
trigger = Trigger(
    tiles_df,
    y_cols,
    y_pred_cols,
    thresholds=thresholds,
    trigger_type="focus",
    units=units,
    latex_y_cols=latex_y_cols,
)
trigger.run(reset_condition=reset)
merged_anomalies, return_df = trigger.identify_and_merge_triggers(merge_interval=60)
detections_df = trigger.get_detections_df(["MET"])
trigger.save_detections_csv(detections_df=detections_df)

detections_df, filtered_anomalies = trigger.filter_from_catalog(catalog, merged_anomalies, detections_df)
trigger.save_detections_csv(detections_df=detections_df, suffix="_in_catalog")
for x, v in filtered_anomalies.items():
    merged_anomalies[x] = v
trigger.plot_anomalies(filtered_anomalies, return_df, support_vars=["GOES_XRSA_HARD_EARTH_OCCULTED"])
